# Week 7 - Notebook 1: Data Preparation

## Goal
Prepare training data for fine-tuning Llama 3.2:
1. Load dataset from HuggingFace Hub
2. Analyze token distribution
3. Create prompt-completion pairs
4. Upload to HuggingFace Hub

## Time: 10-15 minutes

In [ ]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from src.items import Item
from src.config import config

# Load environment variables
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")

## Display Configuration

In [ ]:
config.display()

## Step 1: Load Dataset

In [ ]:
print(f"Loading dataset: {config.DATASET_NAME}")
train, val, test = Item.from_hub(config.DATASET_NAME)
items = train + val + test

print(f"\n✅ Loaded:")
print(f"   Training: {len(train):,} items")
print(f"   Validation: {len(val):,} items")
print(f"   Test: {len(test):,} items")
print(f"   Total: {len(items):,} items")

## Step 2: Analyze Token Distribution

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {config.BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
print("✅ Tokenizer loaded")

In [ ]:
# Count tokens in summaries
print("Counting tokens in product summaries...")
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

avg_tokens = sum(token_counts) / len(token_counts)
max_tokens = max(token_counts)

print(f"\n📊 Token Statistics:")
print(f"   Average: {avg_tokens:.1f} tokens")
print(f"   Maximum: {max_tokens} tokens")

In [ ]:
# Visualize token distribution
plt.figure(figsize=(15, 6))
plt.title(f"Token Distribution: Avg {avg_tokens:.1f}, Max {max_tokens}")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.axvline(config.MAX_TOKENS, color='red', linestyle='--', label=f'Cutoff: {config.MAX_TOKENS}')
plt.legend()
plt.show()

In [ ]:
# Check how many items will be truncated
truncated = len([count for count in token_counts if count > config.MAX_TOKENS])
truncated_pct = truncated / len(items) * 100

print(f"\n📏 With cutoff of {config.MAX_TOKENS} tokens:")
print(f"   {truncated:,} items will be truncated ({truncated_pct:.1f}%)")
print(f"   {len(items) - truncated:,} items fit within limit ({100-truncated_pct:.1f}%)")

## Step 3: Create Prompt-Completion Pairs

In [ ]:
# Example: Before creating prompts
print("📝 Example item BEFORE prompt creation:")
print(f"\nTitle: {train[0].title}")
print(f"Price: ${train[0].price:.2f}")
print(f"\nSummary:\n{train[0].summary[:200]}...")

In [ ]:
# Create prompts for all items
print("Creating prompt-completion pairs...")

# Training and validation: round prices
for item in tqdm(train + val, desc="Train/Val"):
    item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=True)

# Test: keep exact prices
for item in tqdm(test, desc="Test"):
    item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)

print("✅ Prompts created")

In [ ]:
# Example: After creating prompts
print("📝 Example item AFTER prompt creation:")
print(f"\n{'='*60}")
print("PROMPT:")
print(f"{'='*60}")
print(test[0].prompt)
print(f"\n{'='*60}")
print("COMPLETION:")
print(f"{'='*60}")
print(test[0].completion)
print(f"{'='*60}")

In [ ]:
# Analyze prompt+completion token counts
print("Counting tokens in full prompts...")
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

avg_prompt_tokens = sum(prompt_token_counts) / len(prompt_token_counts)
max_prompt_tokens = max(prompt_token_counts)

print(f"\n📊 Full Prompt Token Statistics:")
print(f"   Average: {avg_prompt_tokens:.1f} tokens")
print(f"   Maximum: {max_prompt_tokens} tokens")

In [ ]:
# Visualize full prompt token distribution
plt.figure(figsize=(15, 6))
plt.title(f"Full Prompt Token Distribution: Avg {avg_prompt_tokens:.1f}, Max {max_prompt_tokens}")
plt.xlabel('Number of tokens (prompt + completion)')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.axvline(config.MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'Max Seq Length: {config.MAX_SEQ_LENGTH}')
plt.legend()
plt.show()

## Step 4: Upload to HuggingFace Hub

In [ ]:
# Upload prompt-completion dataset
print(f"Uploading to: {config.PROMPTS_DATASET_NAME}")
print("This may take a few minutes...")

Item.push_prompts_to_hub(config.PROMPTS_DATASET_NAME, train, val, test)

print(f"\n✅ Dataset uploaded successfully!")
print(f"\n🔗 View at: https://huggingface.co/datasets/{config.PROMPTS_DATASET_NAME}")

## Summary

✅ Data preparation complete!

**What we did:**
1. Loaded dataset from HuggingFace Hub
2. Analyzed token distribution (avg ~60 tokens)
3. Set cutoff at 110 tokens (truncates ~5% of items)
4. Created structured prompt-completion pairs
5. Uploaded training data to HuggingFace Hub

**Next step:** `02_base_model_test.ipynb` - Test base Llama before fine-tuning